# RigidHitch — fine-tune DINOv2 on the catalog

The embedding notebook runs DINOv2 **frozen**. This one **trains** it, which is the
one change left that can move accuracy.

Why: nyris benchmarked nine off-the-shelf models on fine-grained industrial parts and
all were weak — DINOv3 scores 26.4 R@1 on their fastener catalog against 63.4 for the
same architecture trained on the domain. Our own test of dinov2-large gained 1.7 points
for 2.8x the query cost. So the model is not the problem; the training is.

**The number to beat** (current model, scored only on held-out products):

| slice | queries | top-1 | top-5 |
|---|---|---|---|
| 3+ photos | 1,068 | 35.4% | 68.5% |

Training uses the **tuning split only**. The test half stays unseen so the comparison
measures retrieval rather than memorisation. Budget 1–2 hours on a T4.


## 1 — Confirm a GPU is attached

On CPU this takes days, not hours. Fix the runtime before going further.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.'
)
print(torch.cuda.get_device_name(0))


## 2 — Install

Same pinned `transformers` as the embedding run, so the model loads identically.


In [ ]:
!pip install -q transformers==4.46.3 pydantic-settings==2.7.1 python-dotenv==1.0.1


## 3 — Mount Drive and unzip images to local disk

`/content` is local SSD. Training reads every image many times over, so reading them
from Drive's network filesystem would dominate the run.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/index_build'

!mkdir -p /content/images
!unzip -q -o {DRIVE}/rigidhitch_images_518.zip -d /content/images
!find /content/images -name '*.jpg' | wc -l


## 4 - Load the code

`embed_rows.jsonl` and `split.json` must be the **same files** used for the
original build. The split is what makes the before/after honest - a different
one would quietly let training see the products it is later scored on.


In [ ]:
# Same code-zip route as the embedding run - no GitHub token needed.
# Upload the fresh partpilot_code.zip (it contains the new finetune script)
# to MyDrive/index_build/ first, replacing the old one.
DRIVE = '/content/drive/MyDrive/index_build'

!rm -rf /content/code && mkdir -p /content/code
!unzip -q -o {DRIVE}/partpilot_code.zip -d /content/code
%cd /content/code

# split.json decides which products are held back for testing. Same file as
# the original build, or the before/after comparison means nothing.
!mkdir -p /content/build
!cp {DRIVE}/embed_rows.jsonl {DRIVE}/split.json /content/build/
!ls -lh /content/build


## 5 — Smoke test: 50 products, 1 epoch

**Do not skip.** Three minutes here catches a wrong path or a broken checkout before
you spend two hours finding out. Expect the loss to print and a model to be saved;
accuracy after one epoch on 50 products is meaningless and not the point.


In [ ]:
!python scripts/rigidhitch_finetune.py \
    --build-dir /content/build \
    --images /content/images \
    --out /content/smoke-model \
    --limit-skus 50 --epochs 1 --batch-size 32


## 6 — The full run

Checkpoints to Drive so a disconnect does not lose it. Keep the tab open — Colab
reclaims idle sessions.

If loss plateaus early, raise `--epochs`. If train-acc reaches ~100% while the held-out
score below barely moves, it is memorising: lower `--lr-backbone` or add epochs of
stronger augmentation.


In [ ]:
!python scripts/rigidhitch_finetune.py \
    --build-dir /content/build \
    --images /content/images \
    --out {DRIVE}/rigidhitch-dinov2 \
    --epochs 12 --batch-size 32 --workers 2


## 7 — Re-embed every image with the fine-tuned model

The index must be rebuilt from vectors produced by the *same* model that will serve
queries. Mixing fine-tuned index vectors with base-model query vectors would make
every comparison meaningless, with no error to notice.

The output directory is passed as the backend — the resolver reads the model's own
config and records it as `dinov2`, so the index metadata stays consistent.


In [ ]:
# The embed script reads its row manifest from --build-dir, so the same
# embed_rows.jsonl has to be there. It must be the identical file: the build
# step hashes it and refuses to run if it disagrees with the vectors.
!mkdir -p {DRIVE}/build_ft
!cp {DRIVE}/embed_rows.jsonl {DRIVE}/build_ft/

!python scripts/rigidhitch_embed_images.py \
    --images-dir /content/images --build-dir {DRIVE}/build_ft \
    --backend {DRIVE}/rigidhitch-dinov2 --shard-size 2048


## 8 — Bring it home and measure

Download both files into your local `index_build/`, renaming the array to
`embeddings_ft.npy`, then run **both** of these and compare like for like:

```
# before
python scripts/rigidhitch_eval_index.py --embeddings embeddings_base.npy \
    --whiten --whiten-dims 768 --query-split test

# after
python scripts/rigidhitch_eval_index.py --embeddings embeddings_ft.npy \
    --whiten --whiten-dims 768 --query-split test
```

`--query-split test` is not optional. Without it the fine-tuned model is scored on the
products it trained on and will report a large gain that does not exist.

If it wins, rebuild the shipped index:

```
python scripts/rigidhitch_filter_and_build.py --embeddings embeddings_ft.npy
```

and copy `rigidhitch-dinov2/` into the backend so the runtime embeds queries with it too.


In [ ]:
from google.colab import files

files.download(f'{DRIVE}/build_ft/embeddings.npy')
files.download(f'{DRIVE}/build_ft/embeddings.meta.json')
